# 카카오 + VWorld — 대지 경계 기준 최소거리 분석

주소를 넣으면 **필지(대지) 경계 폴리곤**을 가져와서, 주변 지하철역·학교 같은 요소까지의
**경계선 기준 최소거리**를 구합니다.

**왜 경계 기준인가**

보통 "역까지 300m"라고 할 때는 단지 중심점에서 잰 거리입니다. 하지만 대지가 넓으면
중심점과 경계선의 거리 차이가 수십 m 이상 벌어집니다. 실제로 걷는 출발점은 대지 경계(출입구)에
가까우므로, 경계선에서 재는 쪽이 현실에 가깝습니다.

**전체 흐름**

```
주소 ──카카오 geocode──▶ 좌표
                          │
                          ▼
      VWorld GetFeature (LP_PA_CBND_BUBUN, geomFilter=POINT)
                          │
                          ▼
                   필지 폴리곤 (GeoJSON)
                          │
    카카오 카테고리 검색 ──┼──▶ 주변 POI (지하철역·학교·병원 …)
                          ▼
      경계선 ↔ POI 최소거리  (미터 좌표계로 투영 후 계산)
                          │
                          ▼
                  표 + folium 지도
```

**준비물**

| 키 | 발급처 | 비고 |
| --- | --- | --- |
| `KAKAO_REST_API_KEY` | [developers.kakao.com](https://developers.kakao.com) | 좌표·POI 검색 |
| `VWORLD_API_KEY` | [vworld.kr](https://www.vworld.kr) | 필지 경계 |
| `VWORLD_DOMAIN` | 위 키 발급 시 **등록한 URL** | 요청마다 함께 보내야 함 |

VWorld 는 키 발급 시 등록한 URL 을 `domain` 파라미터로 같이 보내야 인증을 통과합니다.
로컬에서 쓸 거라면 `http://localhost` 로 등록해 두면 됩니다.

## 0. 준비

이 노트북은 **`kakao_local_example.ipynb` 와 독립적으로** 돌아갑니다.
필요한 카카오 함수는 여기서 다시 정의합니다.

In [18]:
# 최초 1회만 실행
# !pip install requests python-dotenv pandas folium shapely pyproj

In [19]:
import json
import math
import os

import folium
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

KAKAO_KEY = os.getenv("KAKAO_REST_API_KEY")
VWORLD_KEY = os.getenv("VWORLD_API_KEY")
VWORLD_DOMAIN = os.getenv("VWORLD_DOMAIN", "https://devprofessional.xyz")

KAKAO_BASE = "https://dapi.kakao.com/v2/local"
KAKAO_HEADERS = {"Authorization": f"KakaoAK {KAKAO_KEY}"}
VWORLD_URL = "https://api.vworld.kr/req/data"

print("카카오 키:", bool(KAKAO_KEY))
print("VWorld 키:", bool(VWORLD_KEY))
print("VWorld 도메인:", VWORLD_DOMAIN)

카카오 키: True
VWorld 키: True
VWorld 도메인: https://devprofessional.xyz


## 1. 카카오 — 주소를 좌표로, 주변 POI 수집

필지를 찾으려면 먼저 그 안의 좌표 한 점이 필요합니다. 주소 검색으로 얻습니다.

**코드 설명**

- `geocode()` — 주소 → 좌표. 결과가 없으면 `None`.
- `search_category()` — 업종 코드로 주변 POI 를 가져옵니다. 페이지를 넘겨 최대 45건까지 모읍니다.
- 카카오는 `x`=경도, `y`=위도이고 값이 **문자열**이라 `float()` 변환이 필요합니다.

In [20]:
CATEGORY_GROUP = {
    "지하철역": "SW8", "학교": "SC4", "병원": "HP8", "대형마트": "MT1",
    "편의점": "CS2", "은행": "BK9", "공공기관": "PO3", "문화시설": "CT1",
    "어린이집·유치원": "PS3", "주차장": "PK6", "약국": "PM9", "카페": "CE7",
}


def geocode(address):
    """주소를 좌표로 변환한다."""
    res = requests.get(f"{KAKAO_BASE}/search/address.json",
                       headers=KAKAO_HEADERS, params={"query": address, "size": 1})
    docs = res.json()["documents"]
    if not docs:
        return None

    d = docs[0]
    road = d.get("road_address") or {}
    return {"input": address, "lng": float(d["x"]), "lat": float(d["y"]),
            "road_address": road.get("address_name", ""),
            "address_name": d.get("address_name", "")}


def search_category(code, lng, lat, radius=1500, max_results=45):
    """업종 코드로 주변 POI 를 모은다 (최대 45건)."""
    out = []
    for page in range(1, 4):
        res = requests.get(
            f"{KAKAO_BASE}/search/category.json",
            headers=KAKAO_HEADERS,
            params={"category_group_code": code, "x": lng, "y": lat,
                    "radius": radius, "page": page, "size": 15, "sort": "distance"},
        )
        body = res.json()
        out.extend(body["documents"])
        if body["meta"]["is_end"] or len(out) >= max_results:
            break

    return out[:max_results]

In [21]:
ADDRESS = "경기도 성남시 분당구 판교역로 235"

site = geocode(ADDRESS)
site

{'input': '경기도 성남시 분당구 판교역로 235',
 'lng': 127.108690978068,
 'lat': 37.402048486442,
 'road_address': '경기 성남시 분당구 판교역로 235',
 'address_name': '경기 성남시 분당구 판교역로 235'}

## 2. VWorld — 필지 경계 가져오기

`LP_PA_CBND_BUBUN`(연속지적도)에 `geomFilter=POINT(경도 위도)` 로 질의하면
그 점이 놓인 필지를 돌려줍니다.

**요청 형식**

```
https://api.vworld.kr/req/data
  ?service=data&request=GetFeature&data=LP_PA_CBND_BUBUN
  &key=<인증키>&domain=<등록 URL>
  &geomFilter=POINT(127.108 37.401)&crs=EPSG:4326&format=json&size=10
```

**코드 설명**

- **응답 껍데기가 두 가지** 형태로 올 수 있습니다. 최상위에 `response` 가 있는 경우와 없는 경우,
  `result` 안에 `featureCollection` 이 한 겹 더 있는 경우와 아닌 경우 모두 대응합니다.
- `status` 가 `OK` 가 아니면 그대로 예외를 냅니다. 키·도메인 문제를 조용히 넘기면
  나중에 훨씬 찾기 어려워집니다.
- **`geomFilter` 는 점을 포함하지 않는 인접 필지까지 함께 줄 수 있습니다.**
  그래서 받은 뒤 실제로 그 점을 포함하는 필지를 골라냅니다.
- WKT 좌표 순서는 `POINT(경도 위도)` 입니다. 위경도 순서를 바꾸면 엉뚱한 필지가 나옵니다.

In [22]:
def vworld_get_feature(data, geom_filter, crs="EPSG:4326", size=10, **extra):
    """VWorld 데이터 API GetFeature 호출. features 리스트를 반환한다."""
    if not VWORLD_KEY:
        raise ValueError("VWORLD_API_KEY 가 없습니다. .env 를 확인하세요.")

    params = {
        "service": "data", "request": "GetFeature", "data": data,
        "key": VWORLD_KEY, "domain": VWORLD_DOMAIN,
        "geomFilter": geom_filter, "crs": crs,
        "format": "json", "size": size, **extra,
    }
    res = requests.get(VWORLD_URL, params=params, timeout=10)
    res.raise_for_status()

    body = res.json()
    body = body.get("response", body)          # 껍데기가 있을 수도, 없을 수도 있다

    status = body.get("status")
    if status != "OK":
        raise RuntimeError(f"VWorld 응답 status={status} / {body.get('error')}")

    result = body.get("result") or {}
    fc = result.get("featureCollection", result)   # 한 겹 더 들어있는 경우도 대응
    return fc.get("features", [])

In [23]:
def get_parcel(lng, lat, size=10):
    """좌표가 놓인 필지 하나를 반환한다. 없으면 None."""
    feats = vworld_get_feature("LP_PA_CBND_BUBUN", f"POINT({lng} {lat})", size=size)
    if not feats:
        return None

    # geomFilter 가 인접 필지까지 줄 수 있으므로, 점을 실제로 포함하는 것을 고른다
    from shapely.geometry import Point, shape

    pt = Point(lng, lat)
    for f in feats:
        if shape(f["geometry"]).contains(pt):
            return f

    print(f"경고: {len(feats)}건 중 점을 포함하는 필지가 없어 첫 번째를 사용합니다.")
    return feats[0]

In [24]:
parcel = get_parcel(site["lng"], site["lat"])

print("필지 속성:")
for k, v in parcel["properties"].items():
    print(f"  {k}: {v}")

print()
print("도형 타입:", parcel["geometry"]["type"])

필지 속성:
  gosi_year: 2025
  pnu: 4113510900106810000
  jibun: 681대
  bonbun: 681
  bubun: 
  addr: 경기도 성남시 분당구 삼평동 681
  gosi_month: 01
  jiga: 6118000

도형 타입: MultiPolygon


필지 속성에서 자주 쓰는 것들입니다. 실제 키 이름은 위 출력으로 확인하세요.

| 속성 | 뜻 |
| --- | --- |
| `pnu` | 필지 고유번호(19자리) — 다른 공공데이터와 조인할 때 씀 |
| `jibun` | 지번 |
| `bchk` | 대장 구분 (0=토지, 1=임야) |
| `sgg_oid` | 시군구 내 일련번호 |

`geometry` 는 `Polygon` 또는 `MultiPolygon` 입니다. 필지가 여러 조각으로 나뉜 경우
`MultiPolygon` 이 오므로, 이후 코드는 둘 다 처리합니다.

## 3. 방법 A — pyproj 투영 + shapely

경위도 상태로는 미터 거리를 잴 수 없습니다. **미터 좌표계로 투영한 뒤** 계산합니다.

**어떤 좌표계를 쓸 것인가**

측지선 거리를 기준으로 비교해 본 결과입니다 (5km 이내, 필지 경계 ↔ POI):

| 좌표계 | 최대 오차 |
| --- | --- |
| **AEQD** (필지 중심 방위등거리) | 0.000 m |
| **EPSG:5186** (중부원점 TM) | 0.008 m |
| EPSG:5179 (UTM-K) | 1.920 m |

UTM-K(5179)는 전국 단일 좌표계라 편하지만 축척계수(0.9996) 때문에 수 m 오차가 납니다.
여기서는 **필지 중심을 원점으로 하는 AEQD** 를 씁니다. 대상지마다 원점을 새로 잡으므로
그 주변에서는 사실상 오차가 없습니다.

**코드 설명**

- `make_projector()` — 경위도 ↔ 미터 변환 함수 쌍을 만듭니다. 지도에 다시 그리려면
  역변환도 필요해서 둘 다 돌려줍니다.
- `always_xy=True` — pyproj 는 좌표계에 따라 (위도, 경도) 순서를 쓰기도 합니다.
  이 옵션을 켜야 항상 **(경도, 위도)** 순서로 통일됩니다. 빠뜨리면 좌표가 뒤집힙니다.
- `poly.exterior.distance(pt)` vs `poly.distance(pt)`
  - 전자는 **경계선까지의 거리** — 점이 필지 안에 있어도 양수입니다.
  - 후자는 점이 안에 있으면 **0** 입니다.
  - 지하철역은 대개 필지 밖이라 값이 같지만, 의미가 다르므로 둘 다 계산해 둡니다.
- `MultiPolygon` 은 `exterior` 가 없습니다. 그래서 `.boundary` 를 씁니다 (둘 다 처리됨).

In [25]:
from pyproj import Transformer
from shapely.geometry import Point, mapping, shape
from shapely.ops import nearest_points, transform as sh_transform


def make_projector(lat0, lng0):
    """필지 중심을 원점으로 하는 미터 좌표계 변환 함수 (정방향, 역방향)."""
    aeqd = f"+proj=aeqd +lat_0={lat0} +lon_0={lng0} +datum=WGS84 +units=m +no_defs"
    fwd = Transformer.from_crs("EPSG:4326", aeqd, always_xy=True)
    inv = Transformer.from_crs(aeqd, "EPSG:4326", always_xy=True)
    return (lambda g: sh_transform(fwd.transform, g),
            lambda g: sh_transform(inv.transform, g))


def analyze_shapely(parcel_geojson, pois):
    """필지 경계에서 각 POI 까지의 최소거리를 구한다 (방법 A)."""
    poly_ll = shape(parcel_geojson)
    c = poly_ll.centroid
    to_m, to_ll = make_projector(c.y, c.x)

    poly = to_m(poly_ll)
    boundary = poly.boundary            # Polygon/MultiPolygon 모두 동작

    rows = []
    for p in pois:
        pt_ll = Point(float(p["x"]), float(p["y"]))
        pt = to_m(pt_ll)

        near, _ = nearest_points(boundary, pt)      # 경계선 위의 최근접점
        near_ll = to_ll(near)

        rows.append({
            "name": p["place_name"],
            "category": p.get("category_name", "").split(" > ")[-1],
            "boundary_m": boundary.distance(pt),    # 경계선까지 (내부여도 양수)
            "outside_m": poly.distance(pt),         # 내부면 0
            "center_m": poly.centroid.distance(pt),  # 중심점 기준 (비교용)
            "inside": poly.contains(pt),
            "poi_lng": pt_ll.x, "poi_lat": pt_ll.y,
            "near_lng": near_ll.x, "near_lat": near_ll.y,   # 경계선상의 최근접점
        })

    df = pd.DataFrame(rows).sort_values("boundary_m").reset_index(drop=True)
    return df, {"area_m2": poly.area, "perimeter_m": poly.length,
                "centroid": (c.x, c.y)}

In [26]:
pois = search_category(CATEGORY_GROUP["지하철역"], site["lng"], site["lat"], radius=2000)
print(f"지하철역 {len(pois)}건")

df_a, info = analyze_shapely(parcel["geometry"], pois)

print(f"필지 면적 {info['area_m2']:,.1f} m2 ({info['area_m2'] / 3.3058:,.1f} 평)")
print(f"둘레 {info['perimeter_m']:,.1f} m")
print()
df_a[["name", "boundary_m", "center_m", "inside"]].head(10)

지하철역 6건
필지 면적 9,329.0 m2 (2,822.0 평)
둘레 419.6 m



,name,boundary_m,center_m,inside
0,판교역 신분당선,792.483251,839.955026,False
1,판교역 경강선,808.828729,861.421750,False
2,성남역 경강선,1185.888317,1262.612465,False
3,성남역 GTX-A,1282.377643,1359.500891,False
4,이매역 경강선,1763.307369,1843.775647,False
5,이매역 수인분당선,1776.104269,1856.669762,False


`boundary_m` 과 `center_m` 의 차이가 곧 **"중심점에서 재던 거리"의 과대평가분**입니다.
대지가 넓을수록 이 값이 커집니다.

In [27]:
df_a["차이_m"] = df_a["center_m"] - df_a["boundary_m"]
df_a[["name", "boundary_m", "center_m", "차이_m"]].head(10).round(1)

,name,boundary_m,center_m,차이_m
0,판교역 신분당선,792.5,840.0,47.5
1,판교역 경강선,808.8,861.4,52.6
2,성남역 경강선,1185.9,1262.6,76.7
3,성남역 GTX-A,1282.4,1359.5,77.1
4,이매역 경강선,1763.3,1843.8,80.5
5,이매역 수인분당선,1776.1,1856.7,80.6


## 4. 방법 B — 외부 라이브러리 없이 직접 구현

같은 계산을 shapely·pyproj 없이 해봅니다. 원리를 확인하고, 방법 A 를 검증하는 대조군이 됩니다.

**세 조각으로 나뉩니다**

1. **경위도 → 로컬 평면(m)** — 기준점 주변을 평면으로 근사합니다.
2. **점 ↔ 선분 최소거리** — 폴리곤의 모든 변에 대해 구해 최솟값을 취합니다.
3. **내부 판정** — ray casting.

**코드 설명**

- `deg_scales()` — 위도 1도, 경도 1도의 실제 미터를 위도에 따라 계산합니다.
  흔히 쓰는 **111,320 m 고정값은 위도 37.4°에서 0.3% 어긋납니다**(실제 약 110,985 m).
  2.5km 거리에서 7m 넘게 차이 나므로, 급수 근사식을 씁니다.
- `point_segment_dist()` — 점을 선분에 정사영한 위치 `t` 를 구하고 `0~1` 로 자릅니다.
  자르지 않으면 **선분이 아니라 무한 직선까지의 거리**가 되어 값이 작게 나옵니다.
- `point_in_ring()` — 점에서 오른쪽으로 반직선을 쏴 변과 만나는 횟수를 셉니다.
  홀수면 내부입니다.
- `(ay > py) != (by > py)` — 변의 양 끝이 기준선 위아래로 갈리는지 봅니다.
  이 형태로 쓰면 꼭짓점을 정확히 지날 때 두 번 세는 문제를 피할 수 있습니다.

In [28]:
def deg_scales(lat_deg):
    """위도 1도 / 경도 1도의 실제 길이(m). WGS84 급수 근사."""
    p = math.radians(lat_deg)
    m_lat = (111132.92 - 559.82 * math.cos(2 * p)
             + 1.175 * math.cos(4 * p) - 0.0023 * math.cos(6 * p))
    m_lng = (111412.84 * math.cos(p) - 93.5 * math.cos(3 * p)
             + 0.118 * math.cos(5 * p))
    return m_lat, m_lng


def local_frame(lat0, lng0):
    """기준점 중심의 로컬 평면(m) 변환 함수."""
    k_lat, k_lng = deg_scales(lat0)
    return lambda lng, lat: ((lng - lng0) * k_lng, (lat - lat0) * k_lat)


def point_segment_dist(px, py, ax, ay, bx, by):
    """점 P 에서 선분 AB 까지의 최소거리."""
    dx, dy = bx - ax, by - ay
    if dx == 0 and dy == 0:
        return math.hypot(px - ax, py - ay)

    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)
    t = max(0.0, min(1.0, t))                 # 선분 밖으로 나가지 않게 자른다
    return math.hypot(px - (ax + t * dx), py - (ay + t * dy))


def point_in_ring(px, py, ring):
    """ray casting 내부 판정."""
    inside = False
    for i in range(len(ring)):
        ax, ay = ring[i]
        bx, by = ring[(i + 1) % len(ring)]
        if (ay > py) != (by > py):
            if px < ax + (py - ay) * (bx - ax) / (by - ay):
                inside = not inside
    return inside

In [29]:
def iter_rings(geom):
    """Polygon / MultiPolygon 에서 (외곽선, 구멍들) 링을 모두 꺼낸다."""
    if geom["type"] == "Polygon":
        polys = [geom["coordinates"]]
    elif geom["type"] == "MultiPolygon":
        polys = geom["coordinates"]
    else:
        raise ValueError(f"지원하지 않는 도형: {geom['type']}")

    for poly in polys:
        for ring in poly:
            yield [tuple(c[:2]) for c in ring]


def analyze_manual(parcel_geojson, pois):
    """필지 경계에서 각 POI 까지의 최소거리를 구한다 (방법 B)."""
    rings_ll = list(iter_rings(parcel_geojson))

    pts = [c for ring in rings_ll for c in ring]
    lng0 = sum(c[0] for c in pts) / len(pts)          # 기준점은 필지 중앙 부근
    lat0 = sum(c[1] for c in pts) / len(pts)
    to_xy = local_frame(lat0, lng0)

    rings = []
    for ring in rings_ll:
        r = [to_xy(*c) for c in ring]
        if len(r) > 1 and r[0] == r[-1]:
            r = r[:-1]                                # 닫힌 링의 중복 끝점 제거
        rings.append(r)

    rows = []
    for p in pois:
        px, py = to_xy(float(p["x"]), float(p["y"]))

        best = min(
            point_segment_dist(px, py, *ring[i], *ring[(i + 1) % len(ring)])
            for ring in rings for i in range(len(ring))
        )
        inside = point_in_ring(px, py, rings[0])      # 외곽선 기준

        rows.append({"name": p["place_name"], "boundary_m": best,
                     "outside_m": 0.0 if inside else best, "inside": inside})

    return pd.DataFrame(rows).sort_values("boundary_m").reset_index(drop=True)

In [30]:
df_b = analyze_manual(parcel["geometry"], pois)
df_b.head(10).round(2)

,name,boundary_m,outside_m,inside
0,판교역 신분당선,792.48,792.48,False
1,판교역 경강선,808.83,808.83,False
2,성남역 경강선,1185.85,1185.85,False
3,성남역 GTX-A,1282.34,1282.34,False
4,이매역 경강선,1763.23,1763.23,False
5,이매역 수인분당선,1776.04,1776.04,False


## 5. 두 방법 비교

같은 값이 나와야 합니다. 어긋나면 둘 중 하나가 틀린 것이니 반드시 확인하고 넘어갑니다.

In [31]:
cmp = (df_a[["name", "boundary_m"]].rename(columns={"boundary_m": "A_shapely"})
       .merge(df_b[["name", "boundary_m"]].rename(columns={"boundary_m": "B_직접"}),
              on="name"))
cmp["차이_m"] = (cmp["A_shapely"] - cmp["B_직접"]).abs()
cmp["차이_%"] = cmp["차이_m"] / cmp["A_shapely"] * 100

print(f"최대 차이: {cmp['차이_m'].max():.3f} m ({cmp['차이_%'].max():.4f} %)")
print()
cmp.round(3)

최대 차이: 0.074 m (0.0042 %)



,name,A_shapely,B_직접,차이_m,차이_%
0,판교역 신분당선,792.483,792.482,0.001,0.000
1,판교역 경강선,808.829,808.826,0.003,0.000
2,성남역 경강선,1185.888,1185.854,0.035,0.003
3,성남역 GTX-A,1282.378,1282.337,0.041,0.003
4,이매역 경강선,1763.307,1763.234,0.074,0.004
5,이매역 수인분당선,1776.104,1776.038,0.066,0.004


**차이가 큰 경우 의심할 것**

| 증상 | 원인 |
| --- | --- |
| 전부 일정 비율로 어긋남 | 로컬 평면 기준점이 대상지에서 멀다 — 필지 중앙으로 잡았는지 확인 |
| 특정 POI 만 크게 어긋남 | 그 POI 가 필지 안이거나 경계에 아주 가까움 (`inside` 확인) |
| 값이 몇 배 차이 | 위경도 순서가 바뀌었을 가능성 |

참고로 방법 B 는 **거리가 멀어질수록** 평면 근사 오차가 커집니다.
검증에서는 5km 지점에서 약 0.5m 였습니다. 반경 2km 이내라면 수 cm 수준입니다.

## 6. 지도로 확인

필지 폴리곤, POI, 그리고 **경계선상의 최근접점까지 잇는 선**을 함께 그립니다.
선이 엉뚱한 곳에 붙어 있으면 계산이 잘못된 것이라 눈으로 검증하기 좋습니다.

**코드 설명**

- `folium.GeoJson()` — GeoJSON 을 그대로 넣으면 폴리곤이 그려집니다. 좌표 변환이 필요 없습니다.
- `folium.PolyLine([[위도, 경도], ...])` — folium 은 항상 **[lat, lng]** 순서입니다.
  GeoJSON 은 `[lng, lat]` 이라 순서가 반대이므로 주의해야 합니다.
- `df_a` 의 `near_lng`/`near_lat` 가 방법 A 에서 구한 경계선상의 최근접점입니다.

In [32]:
def make_map(parcel_geojson, df, center, top=5, zoom=16):
    """필지 + POI + 최단거리 선을 그린 지도."""
    m = folium.Map(location=[center["lat"], center["lng"]], zoom_start=zoom,
                   tiles="CartoDB positron")

    folium.GeoJson(
        parcel_geojson,
        name="필지",
        style_function=lambda _: {"color": "#d62728", "weight": 3,
                                  "fillColor": "#d62728", "fillOpacity": 0.15},
    ).add_to(m)

    for _, r in df.head(top).iterrows():
        folium.CircleMarker(
            [r["poi_lat"], r["poi_lng"]], radius=7,
            color="#1f77b4", fill=True, fill_opacity=0.85, weight=1,
            tooltip=f"{r['name']} — 경계에서 {r['boundary_m']:.0f}m",
        ).add_to(m)

        # 경계선상의 최근접점 -> POI (folium 은 [lat, lng] 순서)
        folium.PolyLine(
            [[r["near_lat"], r["near_lng"]], [r["poi_lat"], r["poi_lng"]]],
            color="#1f77b4", weight=2, opacity=0.7, dash_array="6",
            tooltip=f"{r['boundary_m']:.1f}m",
        ).add_to(m)

    folium.LayerControl().add_to(m)
    return m

In [33]:
m = make_map(parcel["geometry"], df_a, site, top=5)
m.save("vworld_parcel_map.html")
m

## 7. 여러 업종을 한 번에

업종별로 **가장 가까운 한 곳**만 뽑으면 입지 요약표가 됩니다.

**코드 설명**

- 업종마다 카카오 검색 → 방법 A 계산 → 1위만 취합니다.
- 반경 안에 하나도 없으면 그 업종은 건너뜁니다 (`continue`).
- 업종 수만큼 호출이 늘어나므로, 필요한 것만 고르세요.

In [34]:
def nearest_by_category(parcel_geojson, center, names, radius=2000):
    """업종별 최근접 시설 한 곳씩 요약한다."""
    rows = []

    for name in names:
        pois = search_category(CATEGORY_GROUP[name], center["lng"], center["lat"],
                               radius=radius)
        if not pois:
            print(f"{name}: 반경 {radius}m 내 없음")
            continue

        df, _ = analyze_shapely(parcel_geojson, pois)
        top = df.iloc[0]
        rows.append({
            "업종": name,
            "최근접 시설": top["name"],
            "경계기준_m": round(top["boundary_m"], 1),
            "중심기준_m": round(top["center_m"], 1),
            "차이_m": round(top["center_m"] - top["boundary_m"], 1),
            "반경내_개수": len(pois),
        })

    return pd.DataFrame(rows)

In [35]:
summary = nearest_by_category(
    parcel["geometry"], site,
    ["지하철역", "학교", "병원", "대형마트", "편의점", "공공기관"],
    radius=2000,
)
summary

,업종,최근접 시설,경계기준_m,중심기준_m,차이_m,반경내_개수
0,지하철역,판교역 신분당선,792.5,840.0,47.5,6
1,학교,성균관대학교 판교캠퍼스,249.1,313.0,64.0,22
2,병원,루이의원,25.5,100.8,75.3,45
3,대형마트,GS더프레시 봇들마을점,632.3,710.9,78.6,10
4,편의점,세븐일레븐 판교점,29.3,4.4,-24.9,45
5,공공기관,판교119안전센터,451.1,530.5,79.4,11


In [36]:
summary.to_csv("vworld_nearest_summary.csv", index=False, encoding="utf-8-sig")
print("저장 완료: vworld_nearest_summary.csv")

저장 완료: vworld_nearest_summary.csv


## 8. 개발지구 조회

필지뿐 아니라 **개발 예정 택지지구**의 경계도 같은 방식으로 가져올 수 있습니다.

| 서비스ID | 이름 | 내용 |
| --- | --- | --- |
| **`LT_C_LHZONE`** | 사업지구경계도 | LH 사업지구(택지개발·공공주택지구 등) **경계 폴리곤** |
| **`LT_C_LHBLPN`** | 토지이용계획도 | 지구 **내부의 획지별 용도**(공동주택·상업·공원 등) |
| `LT_C_UQ129` | 개발진흥지구 | 국토계획법상 개발 유도 지구 |
| `LT_C_UD801` | 개발제한구역 | 그린벨트 — 해제 이력 추적 |
| `LT_C_UQ141` | 국토계획구역 | |

`LT_C_LHZONE` 으로 지구 경계를 잡고, `LT_C_LHBLPN` 으로 그 안의 용도 배치를 보는 조합이 됩니다.

**한계를 먼저 알아 둘 것**

- **지정·고시된 지구만** 나옵니다. 발표 전 후보지는 어떤 공간정보에도 없습니다.
- **갱신 시차**가 있습니다. 최근 지정된 지구는 반영이 늦을 수 있으니 LH·국토부 고시를 함께 보세요.
- **LH 사업지구 기준**입니다. SH·GH 등 지방 공사나 민간 도시개발사업은 빠질 수 있습니다.
- 준공된 지 오래된 지구가 계속 등재되어 있는지는 지역마다 다릅니다. 8-1 부터 실제로 확인하세요.

### 8-1. 레이어 속성 먼저 확인

속성 이름을 모르는 채로 코드를 짜면 `KeyError` 를 만나기 쉽습니다. **한 건만 받아서 먼저 봅니다.**

> **주의**: VWorld 데이터 API 는 `request=GetFeature` **하나만** 받습니다.
> 문서에는 `GetFeatureType` 도 적혀 있지만 실제로 호출하면 거부됩니다.
> ```
> status=ERROR / INVALID_RANGE
> 유효한 파라미터 값의 범위 : [GetFeature]
> ```
> 그래서 스키마 조회 대신 `size=1` 로 한 건 받아 속성 키를 읽는 방식을 씁니다.

2절의 필지(`LP_PA_CBND_BUBUN`)는 `pnu`, `jibun`, `addr`, `jiga` 같은 속성이었는데,
지구 레이어는 다른 이름을 씁니다. 그래서 이후 코드는 속성 이름을 **하드코딩하지 않습니다.**


In [ ]:
def peek_layer(data, lng, lat, buffer=0, size=1):
    """레이어에서 한 건만 받아 도형 타입과 속성 키를 확인한다.

    VWorld 데이터 API 는 request=GetFeature 만 지원하므로
    스키마 조회(GetFeatureType) 대신 이 방법을 쓴다.
    """
    feats = vworld_get_feature(data, f"POINT({lng} {lat})", buffer=buffer, size=size)
    if not feats:
        print(f"{data}: 해당 위치에 결과 없음 (buffer 를 키워 보세요)")
        return None

    f = feats[0]
    print(f"{data}   도형: {f['geometry']['type']}")
    print("속성:")
    for k, v in f["properties"].items():
        print(f"  {k}: {v}")
    return f


In [ ]:
peek_layer("LT_C_LHZONE", site["lng"], site["lat"], buffer=5000)


### 8-2. 주변 사업지구 찾기

`buffer` 파라미터(미터)를 쓰면 점 주변 반경으로 검색됩니다.
필지 조회 때처럼 `geomFilter=POINT(경도 위도)` 를 쓰되 `buffer` 만 더합니다.

**코드 설명**

- `size=100` — 지구는 개수가 적으니 넉넉히 받습니다.
- 지구 폴리곤은 필지보다 훨씬 크므로, 반경을 좁게 잡으면 **지구 안에 있으면서도 안 잡힐 수** 있습니다.
  `buffer` 는 경계까지의 거리 기준이라 넓게 잡는 편이 안전합니다.
- 여러 건이 잡힐 수 있어 **표로 정리**합니다. `대상지포함` 이 `True` 인 지구를 맨 앞에 두므로,
  이후 `zones[0]` 은 항상 "대상지가 속한 지구"(있다면)가 됩니다.
- 속성 이름은 레이어마다 다르므로 **모든 속성을 그대로 컬럼에 붙입니다.** 계산한 값만 앞으로 뺍니다.


In [ ]:
def zones_to_df(zones, center):
    """지구 목록을 면적·대상지 포함 여부와 함께 표로 정리한다."""
    site_poi = [{"place_name": "대상지",
                 "x": str(center["lng"]), "y": str(center["lat"])}]

    rows = []
    for i, z in enumerate(zones):
        df1, info = analyze_shapely(z["geometry"], site_poi)
        r = df1.iloc[0]
        rows.append({
            "_idx": i,
            "면적_ha": round(info["area_m2"] / 10000, 1),
            "대상지포함": bool(r["inside"]),
            "경계까지_m": round(r["boundary_m"], 1),
            **z["properties"],
        })

    df = pd.DataFrame(rows).sort_values(
        ["대상지포함", "경계까지_m"], ascending=[False, True]
    ).reset_index(drop=True)
    return df


def find_development_zones(lng, lat, radius=5000, data="LT_C_LHZONE"):
    """주변 사업지구를 찾아 (정렬된 features, 요약표) 를 반환한다.

    대상지를 포함하는 지구가 맨 앞에 온다.
    """
    feats = vworld_get_feature(data, f"POINT({lng} {lat})", buffer=radius, size=100)
    print(f"{data}: 반경 {radius}m 내 {len(feats)}건")
    if not feats:
        return [], pd.DataFrame()

    df = zones_to_df(feats, {"lng": lng, "lat": lat})
    ordered = [feats[i] for i in df["_idx"]]        # 표와 같은 순서로 재배열
    return ordered, df.drop(columns=["_idx"])


In [ ]:
zones, zones_df = find_development_zones(site["lng"], site["lat"], radius=5000)
zones_df


### 8-3. 지구 경계 기준 거리

3절에서 만든 `analyze_shapely()` 를 **그대로 재사용**합니다.
필지 대신 지구 경계 폴리곤을 넣으면 "지구 경계에서 지하철역까지 몇 m" 가 나옵니다.

> **지구 규모에 따라 해석이 달라집니다.**
> 분당·판교 같은 신도시는 **신도시 전체가 하나의 택지개발지구**입니다
> (분당 약 1,960 ha, 판교 약 930 ha). 이 경우 `boundary_m` 은
> "신도시 가장자리까지의 거리"라서 수 km 로 나오고, 사업지 검토용으로는 쓸모가 적습니다.
>
> - **어느 지구에 속하는지** 확인 → 8-2 표의 `대상지포함`, 8-5 의 `check_layers()`
> - **지구 안의 개발 계획**을 보려면 → 9절 토지이용계획도(획지 단위)
> - 지구 경계 거리가 의미 있는 경우 → 지구 **밖**에 있는 대상지가 편입 가능성을 볼 때,
>   또는 소규모 지구(수십 ha)일 때

대상지가 지구 **안**에 있으면 `inside=True` 가 되고 `outside_m` 은 0 이 됩니다.
이때 `boundary_m` 은 "지구 가장자리에서 얼마나 안쪽인지"를 뜻합니다.


In [ ]:
if zones:
    zone = zones[0]          # 8-2 에서 정렬해 뒀으므로 대상지가 속한 지구(있다면)가 맨 앞

    site_poi = [{"place_name": "대상지", "x": str(site["lng"]), "y": str(site["lat"])}]
    df_site, zone_info = analyze_shapely(zone["geometry"], site_poi)
    r = df_site.iloc[0]

    print(f"지구 면적 {zone_info['area_m2']:,.0f} m2 "
          f"({zone_info['area_m2'] / 10000:,.1f} ha)")
    print(f"대상지가 지구 안에 있는가: {r['inside']}")
    print(f"지구 경계선까지: {r['boundary_m']:,.1f} m")

    if zone_info["area_m2"] / 10000 > 300:
        print("  ↑ 지구가 300ha 를 넘습니다. 신도시 단위라 경계 거리는 참고용입니다 (9절 참고).")
    print()

    df_zone, _ = analyze_shapely(zone["geometry"], pois)
    display(df_zone[["name", "boundary_m", "center_m", "inside"]].round(1).head())
else:
    print("반경 내 사업지구가 없습니다. radius 를 늘려 보세요.")


### 8-4. 지구 + 필지 겹쳐 그리기

세 겹을 한 지도에 올립니다 — 사업지구 경계, 토지이용계획(획지), 대상 필지.

**코드 설명**

- `folium.GeoJson()` 은 **FeatureCollection** 을 통째로 받아 여러 도형을 한 번에 그립니다.
- **`GeoJsonPopup(fields=...)`** — 클릭하면 속성이 뜹니다. 속성 이름을 모르므로
  첫 feature 의 키에서 자동으로 뽑습니다. 하드코딩하면 레이어가 바뀔 때 깨집니다.
- `color_by` 에 속성 이름을 주면 그 값별로 색을 나눕니다. 토지이용계획도의 용도 구분에 씁니다.
  8-1 / 8-5 출력에서 실제 속성 이름을 확인한 뒤 넣으세요.
- `LayerControl` 로 겹을 껐다 켜야 아래층이 보입니다.

In [ ]:
BLOCK_COLORS = ["#e6550d", "#3182bd", "#31a354", "#756bb1", "#e377c2",
                "#8c564b", "#17becf", "#d62728", "#bcbd22", "#7f7f7f"]


def _fc(features):
    return {"type": "FeatureCollection", "features": features}


def _popup(features, limit=6):
    """첫 feature 의 속성 키로 팝업 필드를 만든다 (이름 하드코딩 회피)."""
    if not features:
        return None
    fields = [k for k, v in features[0]["properties"].items() if v is not None][:limit]
    return folium.GeoJsonPopup(fields=fields) if fields else None


def make_zone_map(center, parcel_geojson=None, zones=None, blocks=None,
                  color_by=None, zoom=13):
    """사업지구 + 토지이용계획 + 필지를 한 지도에 겹쳐 그린다."""
    m = folium.Map(location=[center["lat"], center["lng"]], zoom_start=zoom,
                   tiles="CartoDB positron")

    if blocks:
        values = (sorted({str(f["properties"].get(color_by, "")) for f in blocks})
                  if color_by else [])
        cmap = {v: BLOCK_COLORS[i % len(BLOCK_COLORS)] for i, v in enumerate(values)}

        folium.GeoJson(
            _fc(blocks), name="토지이용계획",
            style_function=lambda feat: {
                "color": "#555555", "weight": 0.6, "fillOpacity": 0.55,
                "fillColor": cmap.get(str(feat["properties"].get(color_by, "")), "#999999"),
            },
            popup=_popup(blocks),
        ).add_to(m)

        if cmap:
            print("용도 색상:", dict(list(cmap.items())[:10]))

    if zones:
        folium.GeoJson(
            _fc(zones), name="사업지구 경계",
            style_function=lambda _: {"color": "#1f77b4", "weight": 3,
                                      "fillOpacity": 0.05, "fillColor": "#1f77b4"},
            popup=_popup(zones),
        ).add_to(m)

    if parcel_geojson:
        folium.GeoJson(
            parcel_geojson, name="대상 필지",
            style_function=lambda _: {"color": "#d62728", "weight": 3,
                                      "fillOpacity": 0.35, "fillColor": "#d62728"},
        ).add_to(m)

    folium.Marker([center["lat"], center["lng"]], tooltip="대상지",
                  icon=folium.Icon(color="red", icon="star")).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    return m

In [ ]:
# 토지이용계획도 (지구 내부 획지)
blocks = vworld_get_feature(
    "LT_C_LHBLPN", f"POINT({site['lng']} {site['lat']})", buffer=3000, size=500
)
print(f"토지이용계획 {len(blocks)}건")

if blocks:
    print("속성 예시:", blocks[0]["properties"])

In [ ]:
# color_by 는 위 출력에서 용도 구분에 해당하는 속성 이름으로 바꾸세요
zone_map = make_zone_map(
    site,
    parcel_geojson=parcel["geometry"],
    zones=zones,
    blocks=blocks,
    color_by=None,
)
zone_map.save("vworld_zone_map.html")
zone_map

### 8-5. 다른 레이어도 같은 방식으로

`vworld_get_feature()` 는 레이어 ID 만 바꾸면 그대로 동작합니다.
여러 레이어를 훑어 대상지가 어느 규제·지구에 걸리는지 한 번에 확인할 수 있습니다.

**코드 설명**

- 없는 레이어 ID 나 권한이 없는 레이어는 예외를 던지므로 `try` 로 감싸 계속 진행합니다.
  하나가 실패해도 나머지 결과는 봐야 하기 때문입니다.
- **`buffer=0`** 은 그 점이 **실제로 걸치는** 도형만 찾습니다. 규제 확인에는 이쪽이 맞습니다.
  주변에 뭐가 있는지 보려면 `buffer` 를 키우세요.

In [ ]:
CHECK_LAYERS = {
    "LT_C_LHZONE": "사업지구경계",
    "LT_C_LHBLPN": "토지이용계획",
    "LT_C_UQ129": "개발진흥지구",
    "LT_C_UD801": "개발제한구역",
    "LT_C_UQ111": "도시지역",
    "LT_C_UQ141": "국토계획구역",
}


def check_layers(lng, lat, layers=CHECK_LAYERS, buffer=0):
    """대상지가 어느 지구·구역에 걸치는지 확인한다."""
    rows = []

    for data, label in layers.items():
        try:
            feats = vworld_get_feature(data, f"POINT({lng} {lat})",
                                       buffer=buffer, size=10)
            hit = len(feats) > 0
            note = str(feats[0]["properties"])[:70] if hit else ""
        except Exception as e:
            hit, note = None, f"조회 실패: {type(e).__name__} {e}"

        rows.append({"레이어": data, "이름": label, "해당": hit, "비고": note})

    return pd.DataFrame(rows)


check_layers(site["lng"], site["lat"])

## 9. 토지이용계획도 — 지구 안의 획지별 용도

8절의 사업지구 경계(`LT_C_LHZONE`)는 신도시 전체를 통째로 잡습니다.
**`LT_C_LHBLPN`(토지이용계획도)** 은 그 안을 **획지 단위**로 쪼개어 용도를 알려줍니다.

| 궁금한 것 | 쓸 레이어 |
| --- | --- |
| 이 땅이 어느 지구에 속하나 | `LT_C_LHZONE` (8절) |
| **이 땅의 계획상 용도가 뭔가** | **`LT_C_LHBLPN` (9절)** |
| 주변에 공동주택·상업·공원이 어떻게 배치됐나 | `LT_C_LHBLPN` |

**주의**: 토지이용계획도는 **지구 지정 당시의 계획**입니다. 준공 후 변경되었을 수 있고,
지구 밖에서는 아무것도 나오지 않습니다.

### 9-1. 획지 가져오기

획지는 수백 개가 나올 수 있어 **페이지를 넘겨가며** 모읍니다.
VWorld 는 `size` 최대 1000, `page` 로 다음 장을 받습니다.

**코드 설명**

- `vworld_get_all()` — 받은 개수가 `page_size` 보다 적으면 마지막 장이므로 멈춥니다.
- `max_pages` 는 안전장치입니다. 조건을 잘못 줘서 수만 건을 긁는 사고를 막습니다.
- `vworld_get_feature()` 는 `**extra` 로 넘긴 값을 그대로 파라미터에 실으므로 `page` 가 전달됩니다.

In [ ]:
def vworld_get_all(data, geom_filter, buffer=0, page_size=1000, max_pages=10, **kw):
    """페이지를 넘겨가며 전부 모은다."""
    out = []

    for page in range(1, max_pages + 1):
        feats = vworld_get_feature(data, geom_filter, buffer=buffer,
                                   size=page_size, page=page, **kw)
        out.extend(feats)
        if len(feats) < page_size:
            break
    else:
        print(f"경고: {max_pages}페이지에서 중단했습니다. 더 있을 수 있습니다.")

    return out

In [ ]:
blocks = vworld_get_all(
    "LT_C_LHBLPN", f"POINT({site['lng']} {site['lat']})", buffer=1500
)
print(f"획지 {len(blocks)}건")

if blocks:
    peek_layer("LT_C_LHBLPN", site["lng"], site["lat"], buffer=1500)

### 9-2. 용도 컬럼 찾기

속성 이름이 레이어·시기마다 달라서 하드코딩할 수 없습니다.
**용도 구분처럼 보이는 컬럼을 자동으로 추정**하고, 결과를 보고 직접 지정할 수 있게 합니다.

**코드 설명**

- 판단 기준은 세 가지입니다 — 값이 **문자열**이고, **고유값이 2~40개**이며(1개면 구분이 안 되고
  너무 많으면 이름·번호), **숫자만으로 이루어지지 않은** 것.
- 후보를 전부 출력하므로, 자동 추정이 틀렸으면 눈으로 보고 고르면 됩니다.

In [ ]:
from collections import Counter


def guess_use_column(blocks, min_uniq=2, max_uniq=40, max_ratio=0.6):
    """용도 구분에 해당할 만한 속성 컬럼 후보를 찾는다.

    핵심은 '반복되는가' 이다. 용도는 여러 획지가 같은 값을 공유하지만,
    획지번호 같은 ID 는 값이 전부 다르다. 고유값 비율(ratio)로 걸러낸다.
    """
    if not blocks:
        return None, {}

    n = len(blocks)
    cands = {}

    for k in blocks[0]["properties"].keys():
        vals = [str(f["properties"].get(k, "")).strip() for f in blocks]
        vals = [v for v in vals if v and v.lower() != "none"]
        if not vals:
            continue

        uniq = set(vals)
        ratio = len(uniq) / len(vals)          # 1.0 이면 전부 고유 -> ID 성격

        if not (min_uniq <= len(uniq) <= max_uniq):
            continue
        if ratio > max_ratio:
            continue                            # 거의 전부 고유하면 용도가 아니다
        if all(v.replace(".", "").replace("-", "").isdigit() for v in uniq):
            continue                            # 숫자만이면 코드·번호일 가능성

        cands[k] = {"uniq": len(uniq), "ratio": ratio,
                    "top": Counter(vals).most_common(5)}

    if not cands:
        print(f"후보 없음 (획지 {n}건). 속성 목록을 직접 보고 USE_COL 을 지정하세요.")
        print("  속성:", list(blocks[0]["properties"].keys()))
        return None, {}

    for k, c in sorted(cands.items(), key=lambda kv: kv[1]["ratio"]):
        print(f"  {k}: 고유값 {c['uniq']}개 (고유비율 {c['ratio']:.2f}) / 상위 {c['top']}")

    # 반복이 많은 것(고유비율이 낮은 것) 우선
    best = min(cands, key=lambda k: (cands[k]["ratio"], cands[k]["uniq"]))
    print()
    print("자동 선택:", best)
    return best, cands


USE_COL, _cands = guess_use_column(blocks)


In [ ]:
# 자동 추정이 틀렸으면 여기서 직접 지정하세요
# USE_COL = "lnd_use"
print("사용할 용도 컬럼:", USE_COL)

### 9-3. 대상지가 속한 획지

대상지 좌표를 포함하는 획지를 찾습니다. 계획상 이 땅이 무슨 용도인지 바로 나옵니다.

**코드 설명**

- 8-2 의 지구 선택과 같은 방식입니다 — `shapely` 로 점 포함 여부를 봅니다.
- 획지가 겹치는 경우가 있어 **모두** 반환합니다. 보통은 1건입니다.

In [ ]:
from shapely.geometry import Point as _Point, shape as _shape


def blocks_containing(blocks, lng, lat):
    """좌표를 포함하는 획지를 모두 찾는다."""
    pt = _Point(lng, lat)
    return [f for f in blocks if _shape(f["geometry"]).contains(pt)]


hits = blocks_containing(blocks, site["lng"], site["lat"])
print(f"대상지를 포함하는 획지 {len(hits)}건")

for f in hits:
    print()
    for k, v in f["properties"].items():
        if v not in (None, ""):
            print(f"  {k}: {v}")

### 9-4. 용도별 면적 집계

획지 면적을 합쳐 지구 안의 용도 구성을 봅니다.

**코드 설명**

- 면적은 **3절의 `make_projector()`(AEQD)** 로 투영해 계산합니다. 경위도 그대로는 면적이 나오지 않습니다.
- 기준점은 전체 획지의 중앙 부근으로 한 번만 잡고, 모든 획지에 **같은 투영**을 씁니다.
  획지마다 다른 원점을 쓰면 면적을 서로 비교할 수 없습니다.
- `MultiPolygon` 도 `shape()` 가 그대로 처리합니다.

In [ ]:
def blocks_to_df(blocks, use_col=None):
    """획지를 면적과 함께 DataFrame 으로 만든다."""
    if not blocks:
        return pd.DataFrame()

    geoms = [_shape(f["geometry"]) for f in blocks]
    all_pts = [g.centroid for g in geoms]
    lng0 = sum(p.x for p in all_pts) / len(all_pts)
    lat0 = sum(p.y for p in all_pts) / len(all_pts)
    to_m, _ = make_projector(lat0, lng0)         # 전체에 같은 투영을 적용

    rows = []
    for f, g in zip(blocks, geoms):
        gm = to_m(g)
        rows.append({
            "용도": str(f["properties"].get(use_col, "")) if use_col else "",
            "면적_m2": gm.area,
            "면적_평": gm.area / 3.3058,
            **{k: v for k, v in f["properties"].items()},
        })

    return pd.DataFrame(rows)


blocks_df = blocks_to_df(blocks, USE_COL)
print(f"획지 {len(blocks_df)}건, 총면적 {blocks_df['면적_m2'].sum() / 10000:,.1f} ha")
blocks_df.head()

In [ ]:
if USE_COL:
    agg = (blocks_df.groupby("용도")
           .agg(획지수=("면적_m2", "size"), 면적_m2=("면적_m2", "sum"))
           .sort_values("면적_m2", ascending=False))
    agg["면적_ha"] = (agg["면적_m2"] / 10000).round(2)
    agg["비율_%"] = (agg["면적_m2"] / agg["면적_m2"].sum() * 100).round(1)

    display(agg[["획지수", "면적_ha", "비율_%"]])
else:
    print("USE_COL 이 정해지지 않아 집계를 건너뜁니다. 9-2 를 확인하세요.")

### 9-5. 용도별 색으로 지도

8-4 의 `make_zone_map()` 에 `color_by` 로 용도 컬럼을 넘기면 획지가 용도별로 칠해집니다.
대상 필지를 함께 올리면 계획상 어떤 용도 구역에 놓였는지 눈으로 확인됩니다.

In [ ]:
land_use_map = make_zone_map(
    site,
    parcel_geojson=parcel["geometry"],
    zones=None,                 # 거대한 지구 경계는 빼고 획지만 본다
    blocks=blocks,
    color_by=USE_COL,
    zoom=15,
)
land_use_map.save("vworld_land_use_map.html")
land_use_map

In [ ]:
blocks_df.to_csv("vworld_land_use_blocks.csv", index=False, encoding="utf-8-sig")
print("저장 완료: vworld_land_use_blocks.csv")

**결과가 0건이라면**

| 원인 | 확인 |
| --- | --- |
| 대상지가 LH 사업지구 밖 | 8-2 의 `대상지포함` 이 `False` 인지 |
| `buffer` 가 너무 좁음 | 3000~5000 으로 늘려 보기 |
| 해당 지구의 계획도가 미등재 | 8-5 `check_layers()` 에서 `LT_C_LHBLPN` 이 `False` 인지 |

---
## 참고

**좌표 순서 정리** — 라이브러리마다 달라서 가장 실수가 잦은 부분입니다.

| 대상 | 순서 |
| --- | --- |
| 카카오 API (`x`, `y`) | 경도, 위도 |
| GeoJSON `coordinates` | 경도, 위도 |
| VWorld `geomFilter` WKT | `POINT(경도 위도)` |
| shapely `Point(...)` | 경도, 위도 (직접 그렇게 넣는 경우) |
| **folium** | **위도, 경도** ← 반대 |
| pyproj | `always_xy=True` 면 경도, 위도 |

**거리 정확도** (측지선 기준, 5km 이내에서 확인)

| 방식 | 최대 오차 |
| --- | --- |
| AEQD + shapely (방법 A) | 0.000 m |
| EPSG:5186 (중부원점 TM) | 0.008 m |
| 직접 구현 + 위도별 정확 상수 (방법 B) | 0.523 m |
| EPSG:5179 (UTM-K) | 1.920 m |
| 위도 1도 = 111,320 m 고정 | 7.447 m |

**자주 겪는 문제**

- VWorld `status=ERROR` → 키 또는 `domain` 불일치. 발급 시 등록한 URL 과 정확히 같아야 합니다.
- 필지가 안 나옴 → 좌표가 도로·하천 위일 수 있습니다. 건물 쪽으로 조금 옮겨 보세요.
- 폴리곤이 지도에서 엉뚱한 곳에 그려짐 → 위경도 순서가 바뀐 경우입니다.
- `MultiPolygon` 에서 `exterior` 오류 → `.boundary` 를 쓰세요.

**다음 단계로 할 만한 것**

- 인접 필지 병합 (아파트 단지처럼 여러 필지로 된 대지)
- 도로 경계까지의 거리 (접도 조건)
- 직선거리 대신 **보행 경로 거리** — 카카오/네이버 길찾기 API 필요